In [ ]:
from pymilvus import MilvusClient

# 实例化一个客户端对象
client = MilvusClient(
    uri="http://192.168.142.128:19530",
)

client.list_databases()

In [ ]:
client.use_database(db_name="default")
client.list_collections()

In [ ]:
client.create_database(db_name="ai0824")

## 使用pymilvus SDK连接Milvus库

In [ ]:
client.list_databases()

In [ ]:
# 直接把链接的uri，数据库名写入代码之中叫做硬编码

client1 = MilvusClient(
    uri="http://192.168.142.128:19530",
    db_name="ai0824"
)

client1.list_collections()

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.getenv("MILVUS_URI")

In [ ]:
from rag_examples.milvus_config import MILVUS_URI, MILVUS_DB_NAME, get_milvus_client
# 封装了配置信息后的获取client的方法

# 第一种
c1 = MilvusClient(
    uri = MILVUS_URI,
    db_name=MILVUS_DB_NAME
)

c1.describe_database(db_name=MILVUS_DB_NAME)

# 第二种方式，通过封装好的工厂方法获取client实例
c2 = get_milvus_client()
c2.describe_database(db_name=MILVUS_DB_NAME)
c2.get_server_version()

# 在Milvus 中创建collection


In [ ]:
from rag_examples.milvus_config import DEFAULT_DIMENSION, DEFAULT_METRIC_TYPE

client = MilvusClient(
    uri = MILVUS_URI,
    db_name = MILVUS_DB_NAME
)

# 创建collection
collection_name = "demo1"

client.create_collection(
    collection_name=collection_name,
    # 向量字段的维度，跟在对文本进行向量化时的维度保持一致
    dimension=DEFAULT_DIMENSION,
    auto_id=True
)

client.describe_collection(collection_name)

In [13]:
from pymilvus import FieldSchema, DataType, CollectionSchema
# 创建自定义字段的collection
# 设定表需要的字段
fields = [
    # 标量字段
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    # 向量字段
    FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=DEFAULT_DIMENSION),
    # 标量字段,每一个chunk的具体的文本内容
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=1024, description="chunk的文本内容"),
    # chunkid：
    FieldSchema(name="chunkid", dtype=DataType.INT16, description="chunk的id"),
    # 文件名
    FieldSchema(name="filename", dtype=DataType.VARCHAR, max_length=100, description="文件名"),
    # 文件的路径
    FieldSchema(name="filepath", dtype=DataType.VARCHAR, max_length=100, description="文件的路径")
]

# 实例化collectionschema
schema = CollectionSchema(fields, "custom collection")

# 选择数据库
client.use_database(db_name=MILVUS_DB_NAME)

collection_name = "custom_collection"

# 创collection
client.create_collection(
    collection_name=collection_name,
    dimension=DEFAULT_DIMENSION,
    metric_type=DEFAULT_METRIC_TYPE,
    schema=schema,
    db_name=MILVUS_DB_NAME
)

In [14]:
from pymilvus.milvus_client import IndexParams
# 向collection中插入数据
from pymilvus import Index
# 准备数据
datas = [
    {
        "vector": [0.1] * DEFAULT_DIMENSION,
        "text": "人工智能时代最重要的技能是持续学习",
        "chunkid": 1,
        "filename": "ai_des.txt",
        "filepath": "c:\\ai_des.txt"
    },
    {
        "vector": [0.2] * DEFAULT_DIMENSION,
        "text": "持续学习是人工智能时代最重要的技能",
        "chunkid": 2,
        "filename": "ai_des.txt",
        "filepath": "c:\\ai_des.txt"
    },
    {
        "vector": [0.3] * DEFAULT_DIMENSION,
        "text": "学习是人工智能时代最重要的技能",
        "chunkid": 3,
        "filename": "ai_des.txt",
        "filepath": "c:\\ai_des.txt"
    }
]

# 做insert
client.insert(
    collection_name=collection_name,
    data=datas
)

# 创建索引
index = IndexParams()
index.add_index(field_name="vector", index_type="FLAT", metric_type=DEFAULT_METRIC_TYPE)
client.create_index(collection_name=collection_name, index_params=index)
# 加载collection
client.load_collection(collection_name=collection_name)

In [19]:
# 在Milvus2.4版本后，增加了一个动态字段的功能

# 设定表需要的字段
fields = [
    # 标量字段
    FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
    # 向量字段
    FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=DEFAULT_DIMENSION),
    # 标量字段,每一个chunk的具体的文本内容
    FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=1024, description="chunk的文本内容"),
    # chunkid：
    FieldSchema(name="chunkid", dtype=DataType.INT16, description="chunk的id"),
    # 文件名
    FieldSchema(name="filename", dtype=DataType.VARCHAR, max_length=100, description="文件名"),
    # 文件的路径
    FieldSchema(name="filepath", dtype=DataType.VARCHAR, max_length=100, description="文件的路径")
]

collection_name = "dynamic_collection"

client.drop_collection(collection_name=collection_name)

schema = CollectionSchema(fields=fields,
    enable_dynamic_field=True)

client.create_collection(
    collection_name=collection_name,
    dimension=DEFAULT_DIMENSION,
    metric_type=DEFAULT_METRIC_TYPE,
    schema=schema,
    db_name=MILVUS_DB_NAME
)


datas = [
    {
        "vector": [0.4] * DEFAULT_DIMENSION,
        "text": "alskdjlaks;ldkjl",
        "chunkid": 1,
        "filename": "ai_des.txt",
        "filepath": "c:\\ai_des.txt",
        "title": "第一章"
    },
    {
        "vector": [0.4] * DEFAULT_DIMENSION,
        "text": "alskdjlaks;ldkjl",
        "chunkid": 1,
        "filename": "ai_des.txt",
        "filepath": "c:\\ai_des.txt",
        "title": "第一章",
        "tag": ["技术文章", "权威高"]
    }
]

# 做insert
client.insert(
    collection_name=collection_name,
    data=datas
)

{'insert_count': 2, 'ids': [469927880758648564, 469927880758648565], 'cost': 0}

In [20]:
# 创建索引
index = IndexParams()
index.add_index(field_name="vector", index_type="FLAT", metric_type=DEFAULT_METRIC_TYPE)
client.create_index(collection_name=collection_name, index_params=index)
# 加载collection
client.load_collection(collection_name=collection_name)